In [0]:
from pyspark.sql.functions import *
fact_orders = spark.table("workspace.gold.fact_orders")

In [0]:
# Total Revenue
kpi_total_revenue = fact_orders.agg(
    round(sum(when(col("status") == "completed", col("revenue_usd"))),2).alias("total_revenue_usd")
)

display(kpi_total_revenue)

kpi_total_revenue.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_total_revenue")


In [0]:
# Revenue by country
kpi_revenue_country = fact_orders.groupBy("country") \
    .agg(round(sum(when(col("status") == "completed", col("revenue_usd"))),2).alias("total_revenue_usd"))

display(kpi_revenue_country)

kpi_revenue_country.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_revenue_country")

In [0]:
# Revenue by channel 
kpi_revenue_channel = fact_orders.groupBy("channel") \
    .agg(round(sum(when(col("status") == "completed", col("revenue_usd"))),2).alias("revenue"))

display(kpi_revenue_channel)

kpi_revenue_channel.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_revenue_channel")

In [0]:
# Completed order count
kpi_completed_orders = fact_orders \
    .filter(col("status") == "completed") \
    .agg(countDistinct("order_id").alias("completed_orders"))

display(kpi_completed_orders)

kpi_completed_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_completed_orders")

In [0]:
# Completed order rate
kpi_completed_order_rate = fact_orders.agg((   
    countDistinct(when(col("status") == "completed", col("order_id"))
    ) / countDistinct("order_id"))
    .alias("completed_order_rate")
)

display(kpi_completed_order_rate)

kpi_completed_order_rate.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_completed_order_rate")

In [0]:
# AOV
kpi_aov = fact_orders.groupBy("order_id") \
    .agg(round(sum(when(col("status") == "completed", col("revenue_usd"))),2).alias("total_revenue_usd")) \
    .agg(round(avg("total_revenue_usd"),2).alias("AOV"))

display(kpi_aov)

kpi_aov.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_aov")

In [0]:
# Top 5 products by revenue
kpi_top_products = fact_orders \
    .groupBy("product_id") \
    .agg(round(sum(when(col("status") == "completed", col("revenue_usd"))),2).alias("revenue_usd")) \
    .orderBy(desc("revenue_usd")) \
    .limit(5)
    
display(kpi_top_products)

kpi_top_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_top_products")

In [0]:
# active customers
kpi_active_customers = fact_orders \
    .select(countDistinct("customer_id").alias("active_customers"))

display(kpi_active_customers)

kpi_active_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_active_customers")

In [0]:
# customer acquistion 
kpi_customer_acquisition = fact_orders \
    .groupBy("customer_id").agg(min("order_date").alias("first_order_date")) \
    .select(date_format(to_date(col("first_order_date"), "dd-MM-yyyy"), "yyyy-MM").alias("year_month"),
        col("customer_id")).groupBy("year_month") \
    .agg(countDistinct("customer_id").alias("new_customers")) \
    .orderBy("year_month")

display(kpi_customer_acquisition)

kpi_customer_acquisition.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_customer_acquisition")

In [0]:
# Data quality score

# kpi_data_quality = silver_orders.agg(
#     (sum(col("is_valid")) / count("*")).alias("data_quality_score")
# )

kpi_data_quality = fact_orders\
    .select((sum(when(
        col("order_id").isNotNull() &       
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &
        col("status").isNotNull(),
        1
    ).otherwise(0)
) / count("*")).alias("data_quality_score"))
    
display(kpi_data_quality)

kpi_data_quality.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_data_quality")